# Delta Table Features and Transaction Logs

Welcome! This demo will teach you how Delta Lake's transaction log enables powerful features like ACID transactions and time travel.

---

## What is Delta Lake?

**Delta Lake** is an open-source storage layer that brings ACID transactions to data lakes.

**Key innovation:** The **Transaction Log** (also called the **Delta Log**)

---

## What is the Transaction Log?

The transaction log is a **ordered record of every transaction** made to a Delta table.

**Think of it like:**
* A ledger in accounting - every change is recorded
* Git history for your data - every commit is tracked
* A journal - chronological record of all operations

**What it stores:**
* Which files were added
* Which files were removed
* Schema changes
* Table properties
* Metadata about each operation

---

## Why Does This Matter?

The transaction log enables:

1. **ACID Transactions** - Reliable, consistent data operations
2. **Time Travel** - Query historical versions of data
3. **Audit Trail** - See who changed what and when
4. **Concurrent Writes** - Multiple writers without conflicts
5. **Schema Evolution** - Track schema changes over time
6. **Rollback** - Undo mistakes by reverting to previous versions

---

## What You'll Learn

1. **How transaction logs work** - Conceptual understanding
2. **DESCRIBE HISTORY** - View the transaction log
3. **ACID properties** - See them in action
4. **Time Travel** - Query historical data
5. **Practical use cases** - Real-world applications

**Let's get started!**

## 1. How Transaction Logs Work

**High-level architecture:**

```
Delta Table
├── _delta_log/              (Transaction log directory)
│   ├── 00000000000.json    (Transaction 0 - CREATE TABLE)
│   ├── 00000000001.json    (Transaction 1 - INSERT)
│   ├── 00000000002.json    (Transaction 2 - UPDATE)
│   ├── 00000000003.json    (Transaction 3 - DELETE)
│   └── ...
└── data/                   (Parquet data files)
    ├── part-00000.parquet
    ├── part-00001.parquet
    └── ...
```

---

## What's in a Transaction Log Entry?

Each JSON file contains:

**1. Add actions** - Files added to the table
```json
{
  "add": {
    "path": "part-00000.parquet",
    "size": 1024,
    "modificationTime": 1234567890,
    "dataChange": true
  }
}
```

**2. Remove actions** - Files removed from the table
```json
{
  "remove": {
    "path": "part-00000.parquet",
    "deletionTimestamp": 1234567890
  }
}
```

**3. Metadata** - Schema and table properties
```json
{
  "metaData": {
    "schemaString": "...",
    "partitionColumns": [...],
    "configuration": {...}
  }
}
```

**4. Protocol** - Delta Lake version

**5. Commit info** - Who, when, what operation

### How Delta Lake Reads Work

**When you query a Delta table:**

1. **Read the transaction log** - Start from the latest transaction
2. **Reconstruct table state** - Replay all transactions to know which files are current
3. **Read only current files** - Skip deleted/old files
4. **Return results** - Query only the active data files

**Example:**

```
Transaction 0: ADD file1.parquet
Transaction 1: ADD file2.parquet
Transaction 2: REMOVE file1.parquet, ADD file3.parquet

Current state: file2.parquet, file3.parquet  ← Only these are read!
```

**Benefits:**
* Always consistent view of data
* No need to physically delete files immediately
* Enables time travel (read old transactions)
* Multiple readers see consistent snapshots

### How Delta Lake Writes Work

**When you write to a Delta table:**

1. **Write data files** - Write new Parquet files to storage
2. **Create transaction entry** - Prepare JSON with add/remove actions
3. **Atomic commit** - Write transaction JSON with next sequence number
4. **Success or fail** - Either entire transaction succeeds or fails

**Example: INSERT operation**

```
Step 1: Write part-00005.parquet (new data)
Step 2: Create transaction JSON:
        {
          "add": {"path": "part-00005.parquet", ...}
        }
Step 3: Atomically write 00000000005.json
Step 4: Transaction committed!
```

**Atomicity guarantee:**
* If step 3 fails, the data file exists but is ignored (orphan file)
* If step 3 succeeds, the transaction is permanent
* No partial transactions - all or nothing!

**Optimistic concurrency:**

Instead of locking the table so only one writer can work at a time, Delta Lake lets multiple writers work **simultaneously** and only checks for conflicts at **commit time**.

It's called *optimistic* because it assumes conflicts are **rare**. Most of the time, two writers touch different parts of the data, so there's no problem. Contrast this with *pessimistic* concurrency (traditional databases), which locks resources upfront "just in case."

* Multiple writers can work simultaneously
* Conflicts detected at commit time
* Failed commits retry automatically

---

**Step-by-step: How it actually works**

```
1. READ current version
   Writer A reads version 5 of the transaction log.
   Writer B also reads version 5.

2. DO WORK (in parallel — no locks!)
   Writer A: Updates rows where region = 'West'
   Writer B: Inserts new rows for region = 'East'
   Both write new Parquet files to storage.

3. COMMIT — this is where the magic happens
   Writer A tries to commit as version 6.
     → Checks: "Is the log still at version 5?" → Yes
     → Writes 00000000006.json → Committed!

   Writer B tries to commit as version 6.
     → Checks: "Is the log still at version 5?" → No, it's now 6!
     → CONFLICT DETECTED

4. AUTOMATIC RETRY
   Writer B re-reads the log (now at version 6).
   Checks: "Does version 6 conflict with my changes?"
     → Version 6 changed 'West' rows, I changed 'East' rows → No conflict!
   Writer B commits as version 7.
```

---

**When does a retry fail?**

If both writers modified the **same rows or files**, the retry detects a true conflict:

```
Writer A: UPDATE products SET price = 10 WHERE id = 1  → Commits as version 6
Writer B: UPDATE products SET price = 20 WHERE id = 1  → Conflict!
  → Re-reads version 6... same file was modified → Cannot safely retry
  → Throws ConcurrentModificationException
```

The application (or user) must decide how to handle this — typically by re-reading and re-applying the logic.

## 2. See Transaction Logs in Action

Let's create a Delta table and perform operations to see how each creates a transaction log entry.

**We'll use DESCRIBE HISTORY to view the transaction log** - no file system access needed!

In [0]:
%sql
-- Create a new Delta table
-- This creates Transaction 0 in the log

CREATE OR REPLACE TABLE workspace.default.products_demo (
  product_id INT,
  product_name STRING,
  category STRING,
  price DOUBLE,
  stock_quantity INT
)
USING DELTA
COMMENT 'Demo table for transaction log exploration'

In [0]:
%sql
-- DESCRIBE HISTORY shows the transaction log!
-- This is how we view transactions without file system access

DESCRIBE HISTORY workspace.default.products_demo

### Understanding DESCRIBE HISTORY Output

**Key columns in the transaction log:**

* **version** - Transaction number (0, 1, 2, 3...)
* **timestamp** - When the transaction occurred
* **userId** - Who made the change
* **userName** - User's email/name
* **operation** - Type of operation (CREATE, INSERT, UPDATE, DELETE, etc.)
* **operationParameters** - Details about the operation
* **readVersion** - Version read before this write
* **isolationLevel** - Transaction isolation level
* **isBlindAppend** - Whether operation only added data

**Common operations you'll see:**
* `CREATE TABLE` / `CREATE OR REPLACE TABLE`
* `WRITE` / `INSERT`
* `UPDATE`
* `DELETE`
* `MERGE`
* `OPTIMIZE`
* `VACUUM`
* `SET TBLPROPERTIES`
* `ADD COLUMNS`

**Key insight:** Every operation creates a new version in the transaction log!

In [0]:
%sql
-- Insert some data
-- This creates a new transaction (Transaction 1)

INSERT INTO workspace.default.products_demo VALUES
  (1, 'Laptop', 'Electronics', 999.99, 50),
  (2, 'Mouse', 'Electronics', 29.99, 200),
  (3, 'Desk', 'Furniture', 299.99, 30),
  (4, 'Chair', 'Furniture', 199.99, 45),
  (5, 'Monitor', 'Electronics', 399.99, 75)

In [0]:
%sql
-- Now we should see 2 transactions:
-- Version 0: CREATE TABLE
-- Version 1: INSERT

DESCRIBE HISTORY workspace.default.products_demo

In [0]:
%sql
-- Update prices (discount on electronics)
-- This creates Transaction 2

UPDATE workspace.default.products_demo
SET price = price * 0.9
WHERE category = 'Electronics'

In [0]:
%sql
-- Now we have 3 transactions
-- Notice the operation type and timestamp

DESCRIBE HISTORY workspace.default.products_demo

In [0]:
%sql
-- Delete out-of-stock items
-- This creates Transaction 3

DELETE FROM workspace.default.products_demo
WHERE stock_quantity < 40

In [0]:
%sql
-- Now we have 4 transactions
-- Each operation is recorded!

DESCRIBE HISTORY workspace.default.products_demo

In [0]:
%sql
-- Add a new column to the table
-- This creates Transaction 4 with schema change

ALTER TABLE workspace.default.products_demo
ADD COLUMN supplier STRING

In [0]:
%sql
-- Transaction log tracks schema evolution too!

DESCRIBE HISTORY workspace.default.products_demo

## 3. ACID Properties in Action

The transaction log enables **ACID guarantees** - let's see each property in action!

### Atomicity - All or Nothing

**What is Atomicity?**

A transaction either **completely succeeds** or **completely fails** - no partial updates.

**How transaction log enables this:**
* Data files are written first
* Transaction JSON is written atomically (last step)
* If JSON write fails, data files are ignored (orphaned)
* If JSON write succeeds, entire transaction is committed

**Example scenario:**
```sql
-- Insert 1 million rows
INSERT INTO table SELECT * FROM large_dataset
```

**Without Delta Lake:**
* If it fails halfway, you have 500K rows (partial data)
* Data is corrupted
* Need to manually clean up

**With Delta Lake:**
* If it fails, transaction log entry is not written
* Table still has 0 rows (no partial data)
* Automatic rollback
* No cleanup needed

**Key Point:** The transaction log entry is the "commit" - without it, data doesn't exist!

In [0]:
%sql
-- Let's insert multiple rows in one transaction
-- Either ALL 3 rows are inserted, or NONE are

INSERT INTO workspace.default.products_demo (product_id, product_name, category, price, stock_quantity) VALUES
  (6, 'Keyboard', 'Electronics', 79.99, 100),
  (7, 'Webcam', 'Electronics', 89.99, 60),
  (8, 'Headphones', 'Electronics', 149.99, 80);

-- Check the data
SELECT * FROM workspace.default.products_demo

In [0]:
%sql
-- This INSERT created ONE transaction
-- All 3 rows are part of the same atomic commit

DESCRIBE HISTORY workspace.default.products_demo
LIMIT 5

### Consistency - Always Valid State

**What is Consistency?**

Data always moves from one **valid state** to another valid state.

**How transaction log enables this:**
* Schema is enforced at write time
* Constraints are validated
* Invalid data is rejected before commit
* Transaction log only records valid transactions

**Result:**
* Transaction is rejected
* No transaction log entry created
* Table reworkspaces in valid state
* No cleanup needed

**Key Point:** Transaction log only contains valid, consistent transactions!

In [0]:
%sql
-- This will FAIL
-- We haven't added in enough data columns

INSERT INTO workspace.default.products_demo VALUES
  (9, 'Desk', 'Furniture', 109.99, 50);

## 4. Time Travel

**What is Time Travel?**

Time Travel lets you query **historical versions** of your Delta table using the transaction log.

**How it works:**
* Transaction log records every version
* Each version is a snapshot of the table at that point in time
* You can query any previous version
* Data files are retained (until VACUUM)

**Use cases:**
* Audit data changes
* Recover from mistakes
* Compare versions
* Reproduce reports
* Debug data issues

Let's see it in action!

In [0]:
%sql
-- First, let's see the current state
SELECT * FROM workspace.default.products_demo
ORDER BY product_id

In [0]:
%sql
-- Review all the versions we created
DESCRIBE HISTORY workspace.default.products_demo

In [0]:
%sql
-- Query Version 1 (right after initial INSERT, before UPDATE)
-- Use VERSION AS OF syntax

SELECT * FROM workspace.default.products_demo VERSION AS OF 1
ORDER BY product_id

In [0]:
%sql
-- Compare current version with Version 1
-- Notice the prices changed (we applied 10% discount in UPDATE)

SELECT 
  'Version 1 (Before Update)' AS version,
  product_id,
  product_name,
  price
FROM workspace.default.products_demo VERSION AS OF 1

UNION ALL

SELECT 
  'Current (After Update)' AS version,
  product_id,
  product_name,
  price
FROM workspace.default.products_demo

ORDER BY product_id, version

In [0]:
%sql
-- You can also query by timestamp
-- Use TIMESTAMP AS OF syntax

SELECT * 
FROM workspace.default.products_demo 
TIMESTAMP AS OF (now() - INTERVAL 10 MINUTE)
ORDER BY product_id

### Time Travel Syntax

**Query by version number:**
```sql
SELECT * FROM table VERSION AS OF 5
SELECT * FROM table@v5
```

**Query by timestamp:**
```sql
SELECT * FROM table TIMESTAMP AS OF '2024-01-15 10:30:00'
SELECT * FROM table@20240115
```

**Query by date:**
```sql
SELECT * FROM table TIMESTAMP AS OF '2024-01-15'
```

**Note:** Time travel only works for versions that haven't been vacuumed!

In [0]:
%sql
-- You can restore a table to a previous version
-- This creates a new transaction that reverts changes

RESTORE TABLE workspace.default.products_demo TO VERSION AS OF 1

In [0]:
%sql
-- Check the data - should be back to Version 1 state
-- Prices should be original (before discount)

SELECT * FROM workspace.default.products_demo
ORDER BY product_id

In [0]:
%sql
-- RESTORE creates a new transaction!
-- The old versions are still in the log

DESCRIBE HISTORY workspace.default.products_demo

## 5. Why and How to Delete Prior Versions with VACUUM

### Why Delete Old Versions?

Every transaction you saw above leaves behind **data files on storage**. Even after an UPDATE or DELETE, the old Parquet files are kept so that time travel still works. Over time this creates a real problem:

* **Storage costs grow** — Old, unreferenced data files accumulate and you pay for every byte.
* **Listing overhead** — Cloud object stores slow down when directories contain thousands of small files.
* **Compliance requirements** — Regulations like GDPR may *require* you to permanently remove deleted records, not just mark them as removed in the log.

### How VACUUM Works

`VACUUM` physically deletes data files that are **no longer referenced** by any transaction log entry within the retention window.

```
┌─────────────────────────────────────────────┐
│  Transaction Log (JSON entries)              │
│  v0 ─► v1 ─► v2 ─► v3 ─► v4 ─► v5 ─► v6   │
│              ▲ retention window ▲             │
│  Files for v0, v1 are UNREFERENCED           │
│  ─► VACUUM removes those Parquet files       │
└─────────────────────────────────────────────┘
```

**Key rules:**
* Default retention period is **7 days** (168 hours).
* Files younger than the retention period are **never** deleted, even if unreferenced.
* After VACUUM, **time travel to versions older than retention will fail** — those data files are gone.
* VACUUM only removes *data* files; the transaction log JSON entries remain.

### VACUUM Syntax

| Command | What It Does |
| --- | --- |
| `VACUUM table` | Delete unreferenced files older than default retention (7 days) |
| `VACUUM table RETAIN 720 HOURS` | Use a custom retention (30 days here) |
| `VACUUM table DRY RUN` | **Preview only** — lists files that *would* be deleted without removing anything |

### Best Practice Workflow

1. **DRY RUN first** — Always preview before deleting.
2. **Set retention deliberately** — Match your business / compliance needs.
3. **Schedule VACUUM** — Run it regularly (e.g., weekly) to keep storage lean.
4. **Coordinate with time travel users** — Make sure nobody relies on versions older than your retention window.

In [0]:
%sql
-- Before vacuuming, inspect the table's physical details
-- numFiles shows how many Parquet files currently exist

DESCRIBE DETAIL workspace.default.products_demo

In [0]:
%sql
-- Set retention to 0 so VACUUM can remove ALL unreferenced files
-- Only do this in demos/dev — never in production!
-- logRetentionDuration  = how long transaction log entries are kept
-- deletedFileRetentionDuration = minimum age before VACUUM can remove a file
ALTER TABLE workspace.default.products_demo
SET TBLPROPERTIES ('delta.deletedFileRetentionDuration' = 'interval 0 hours');

-- DRY RUN shows which files WOULD be deleted
-- With 0-hour retention, this targets all prior version files
VACUUM workspace.default.products_demo DRY RUN

In [0]:
%sql
-- Now actually vacuum — this permanently removes all old version files
VACUUM workspace.default.products_demo RETAIN 0 HOURS

In [0]:
%sql
-- VACUUM itself is recorded as a transaction!
-- Look for the VACUUM operation in the history

DESCRIBE HISTORY workspace.default.products_demo

## 5. Practical Patterns

Real-world patterns using transaction logs.

In [0]:
%sql
-- Create an audit report showing all changes to the table
-- This is perfect for compliance and debugging

SELECT 
  version,
  timestamp,
  operation,
  operationParameters,
  userName,
  CAST(operationMetrics.numOutputRows AS INT) AS rows_affected
FROM (
  DESCRIBE HISTORY workspace.default.products_demo
)
ORDER BY version DESC

In [0]:
%sql
-- Find when a specific product's price changed
-- Compare consecutive versions

WITH version_1 AS (
  SELECT product_id, product_name, price AS price_v1
  FROM workspace.default.products_demo VERSION AS OF 1
),
version_2 AS (
  SELECT product_id, product_name, price AS price_v2
  FROM workspace.default.products_demo VERSION AS OF 2
)
SELECT 
  v1.product_id,
  v1.product_name,
  v1.price_v1 AS original_price,
  v2.price_v2 AS updated_price,
  ROUND(v2.price_v2 - v1.price_v1, 2) AS price_change,
  ROUND((v2.price_v2 - v1.price_v1) * 100.0 / v1.price_v1, 2) AS percent_change
FROM version_1 v1
JOIN version_2 v2 ON v1.product_id = v2.product_id
WHERE v1.price_v1 != v2.price_v2

In [0]:
%sql
-- Check how long data has been retained
-- Useful for planning VACUUM operations

SELECT 
  MIN(timestamp) AS oldest_version,
  MAX(timestamp) AS newest_version,
  COUNT(*) AS total_versions,
  DATEDIFF(MAX(timestamp), MIN(timestamp)) AS retention_days
FROM (
  DESCRIBE HISTORY workspace.default.products_demo
)

In [0]:
%sql
-- Analyze what operations have been performed
-- Understand table usage patterns

SELECT 
  operation,
  COUNT(*) AS operation_count,
  MIN(timestamp) AS first_occurrence,
  MAX(timestamp) AS last_occurrence
FROM (
  DESCRIBE HISTORY workspace.default.products_demo
)
GROUP BY operation
ORDER BY operation_count DESC

## Congratulations!

You've completed the Delta Lake Transaction Logs demo!

### **What You Learned:**

* **Transaction Log Architecture** - How Delta Lake tracks changes  
* **ACID Properties** - Atomicity, Consistency, Isolation, Durability  
* **DESCRIBE HISTORY** - View transaction log without file access  
* **Time Travel** - Query and restore historical versions  
* **Practical Patterns** - Audit trails, debugging, rollback  
* **Best Practices** - Retention, monitoring, performance  

---

### **Key Takeaways:**

1. **Transaction log is the foundation** - Enables all Delta Lake features
2. **Every operation creates a version** - Complete audit trail
3. **ACID guarantees** - Reliable data operations
4. **Time travel is powerful** - Query any historical version
5. **Works in serverless** - No file system access needed
6. **Balance retention vs cost** - Use VACUUM appropriately

---

### **Next Steps:**

* Explore MERGE operations (upserts)
* Learn about Delta Lake optimization (OPTIMIZE, Z-ORDER)
* Study Change Data Feed (CDC)
* Implement data retention policies
* Build production pipelines with Delta Lake

---

### **Resources:**

* [Delta Lake Documentation](https://docs.databricks.com/delta/index.html)
* [Transaction Log Protocol](https://github.com/delta-io/delta/blob/master/PROTOCOL.md)
* [Time Travel Guide](https://docs.databricks.com/delta/history.html)
* [Delta Lake Best Practices](https://docs.databricks.com/delta/best-practices.html)